# 05 – Modellierung & Hyperparameter-Tuning

Dieses Notebook führt den Modellvergleich und das Tuning des finalen LightGBM-Modells durch.

**Schritte:**
1. Modellvergleich (Ridge, RandomForest, XGBoost, LightGBM)
2. Hyperparameter-Tuning mit Optuna (50 Trials, TimeSeriesSplit)
3. Vergleich Standard vs. Log-Transformation der Zielvariable
4. Speichern des finalen Modells

In [ ]:
import sys
import numpy as np
import polars as pl
from pathlib import Path
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate

current_dir = Path('..') / '03_src'
sys.path.append(str(current_dir))
root_dir = Path('..')

from config import target_col, feature_cols, feature_cols_cleaned
from features import (
    add_weekdays, add_weekend, heating_season, renovation_index,
    heating_amount, add_price_features, solar_potentials,
    add_thermal_dynamics, add_cyclic_features, add_building_type_dummies
)
from utilis import train_test_split, correlated_features_drop, compare_models, tune_lightgbm

## 5.1 Daten & Features vorbereiten

In [ ]:
data_combined_cleaned = pl.read_parquet(
    str(root_dir / '02_data' / 'processed' / 'combined_data_cleaned.parquet')
)

df_ml = (
    data_combined_cleaned
    .pipe(add_weekdays).pipe(add_weekend).pipe(heating_season)
    .pipe(renovation_index).pipe(heating_amount).pipe(add_price_features)
    .pipe(solar_potentials).pipe(add_thermal_dynamics)
    .pipe(add_cyclic_features).pipe(add_building_type_dummies)
    .select(['date', target_col] + feature_cols)
)

X_train, X_test, y_train, y_test = train_test_split(
    df_ml, feature_cols=feature_cols, target_col=target_col
)

X_train_red = X_train[feature_cols_cleaned]
X_test_red  = X_test[feature_cols_cleaned]

X_train_final, X_test_final, final_features = correlated_features_drop(
    X_train_red, X_test_red, feautre_cols=feature_cols_cleaned
)
print(f'Finales Feature-Set: {len(final_features)} Features')

## 5.2 Modellvergleich

In [ ]:
comparison = compare_models(X_train_final, X_test_final, y_train, y_test)
print(comparison)

## 5.3 LightGBM Tuning – Standard

In [ ]:
lgbm_model_std, study_std = tune_lightgbm(
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    final_features=final_features,
    output_dir=str(root_dir / '05_plots'),
    n_trials=50, log_transform=False
)

## 5.4 LightGBM Tuning – Log-Transformation

In [ ]:
lgbm_model_log, study_log = tune_lightgbm(
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    final_features=final_features,
    output_dir=str(root_dir / '05_plots'),
    n_trials=50, log_transform=True
)

## 5.5 Modellvergleich Standard vs. Log

In [ ]:
y_pred_std = lgbm_model_std.predict(X_test_final)
y_pred_log = np.expm1(lgbm_model_log.predict(X_test_final))

mae_std = mean_absolute_error(y_test, y_pred_std)
mae_log = mean_absolute_error(y_test, y_pred_log)

print(f'Standard – MAE: {mae_std:.3f} kWh | R²: {r2_score(y_test, y_pred_std):.3f}')
print(f'Log      – MAE: {mae_log:.3f} kWh | R²: {r2_score(y_test, y_pred_log):.3f}')

if mae_log < mae_std:
    print('✅ Log-Transformation ist besser')
    lgbm_model = lgbm_model_log
    y_pred = y_pred_log
else:
    print('✅ Standard ist besser')
    lgbm_model = lgbm_model_std
    y_pred = y_pred_std